In [2]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

paired = pd.read_pickle(
    PROJECT_ROOT / "data/processed/paired_all.pkl"
)

print("Paired shape:", paired.shape)
print("Datasets:", paired["dataset"].nunique())
print("Sources:", paired["source"].nunique())

Paired shape: (18049, 11)
Datasets: 7
Sources: 7


In [3]:
pd.crosstab(paired["dataset"], paired["source"])

source,BigScience,Eleuther-AI,Human,LLAMA,OpenAI,PaLM,Tsinghua
dataset,,,,,,,
cmv,227,184,231,239,239,162,233
eli5,442,363,454,450,461,469,455
sci_gen,351,184,438,446,468,449,389
tldr,355,302,377,360,378,254,367
wp,333,227,269,324,434,94,327
xsum,462,416,342,464,460,438,456
yelp,452,445,486,480,472,473,468


## Task 1 — Data Processing and Version Alignment

We loaded all seven datasets, including Human and six LLM sources. We normalized the version names and aligned T0 with seven T1 paraphraser variants using `key`, `source`, and `dataset`. After excluding incomplete records and one record containing empty text, we obtained 18,049 complete article-source pairs.

In [4]:
samples = paired.groupby("dataset").head(2).copy()

text_cols = [c for c in paired.columns if c.startswith("text_")]

for col in text_cols:
    samples[col] = samples[col].str.slice(0, 80)

display(samples)

,key,source,dataset,text_T0,text_chatgpt,text_palm,text_dipper,text_dipper_low,text_dipper_high,text_pegasus_slight,text_pegasus_full
0,cmv-1079,BigScience,cmv,Please note before reading: Please do not down...,Please be aware that I kindly request not to d...,expressions with Java 8 Stream API The followi...,Before you read any further: I have written a...,Please note before reading: Do not downvote m...,A style of camouflaging the impossible became...,Please note before reading: Please do not down...,Please don't downvote my comments and replies ...
1,cmv-1079,Eleuther-AI,cmv,Please note before reading: Please do not down...,I kindly request that you refrain from downvot...,"**Note:** Before reading, please keep in mind ...","NB: The first time I tried this out, it was b...",Please note before reading: Please do not dow...,"Remember once again I say, downvoting is not ...",Please don't downvote my comments and replies ...,Please don't downvote my comments and replies ...
1515,eli5-16706,BigScience,eli5,It's your body that is preparing to fight for ...,"When faced with a threatening situation, your ...","When confronted with a threatening situation, ...",Because the brain takes it for a threat. Adre...,Because your brain perceives it as a threat. ...,They had gone to watch—a perilous walk—from L...,It's your body that is preparing to fight for ...,"If you see something threatening, your body is..."
1516,eli5-16706,Eleuther-AI,eli5,It's your body that is preparing to fight for ...,When your brain perceives something threatenin...,"When you see something threatening, your body ...","Since the brain takes in the threat, it begin...",Because your brain takes it as a threat. It i...,You must have foreseen some dreadful monster ...,It's your body that is preparing to fight for ...,"If you see something threatening, your body is..."
4609,sci_gen-1744,BigScience,sci_gen,"In this article, we investigate the transient ...",This article explores the behavior of packetsb...,"P: [t 2, t] R will belong to this class. This ...",- It is usually called a factory network. We ...,"In this article, we investigate the transient...",These were three years of great difficulty. T...,"In this article, we investigate the transient ...",Transient behavior of packetsbits traversing a...
4610,sci_gen-1744,Human,sci_gen,"In this article, we investigate whispered-to n...",This article examines a method for converting ...,This paper proposes a modified transformer arc...,It is with this in mind that we propose to in...,"In this article, we investigate a whisper-to-...",Gulathy blew two pitcherfuls of tea for himse...,"In this article, we investigate whispered-to n...",We investigated whispered-to natural-speech co...
7334,tldr-2826,BigScience,tldr,Tensorflow Privacy implements Tensorflow optim...,Tensorflow Privacy is a library that enables t...,information useful for inferring sensitive att...,The author's contribution to the science is a...,Contribution to the research field: TensorFlo...,The next morning we went down to the little t...,Machine learning models with differential priv...,Machine learning models with differential priv...
7335,tldr-2826,Eleuther-AI,tldr,Tensorflow Privacy implements Tensorflow optim...,Tensorflow Privacy provides Tensorflow optimiz...,TensorFlow Privacy provides optimizers for tra...,TensorFlow's privacy group develops a tensorf...,TensorFlow privacy is an implementation of Te...,There was a chance of seeing Amsden's line in...,Machine learning models with differential priv...,Machine learning models with differential priv...
9727,wp-1198,BigScience,wp,"""and those are the terms. I'm sorry."" The Athe...","""Those are the terms, and I apologize,"" said t...",", only 20% of women have access to managerial ...","The Atheist, a young hippie girl in thick gla...","The atheist, a young female hipster in thick-...",Atheism pressed her with even greater persuas...,"""and those are the terms. I'm sorry. The Athei...","Those are the te